# Step 1 — Setup, Imports, Load Dataset, Basic Cleaning



In [ ]:
# Load the Drive helper and mount
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q sentence-transformers faiss-cpu gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 72.0 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np

In [ ]:

# =========================
# LOAD DATASET
# =========================


FILE_PATH = "/content/drive/MyDrive/ITSM_Dataset.csv"

df = pd.read_csv(FILE_PATH)

# =========================
# BASIC CLEANING
# =========================

df.columns = df.columns.str.strip()

for col in df.columns:
    df[col] = df[col].astype(str)

# =========================
# DATASET OVERVIEW
# =========================

print("=" * 50)
print("Dataset Loaded Successfully")
print("=" * 50)

print(f"\nRows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

print("\nColumns:")
for col in df.columns:
    print("-", col)

print("\nSample Data:")
display(df.head(3))

Dataset Loaded Successfully

Rows: 100000
Columns: 22

Columns:
- Status
- Ticket ID
- Priority
- Source
- Topic
- Agent Group
- Agent Name
- Created time
- Expected SLA to resolve
- Expected SLA to first response
- First response time
- SLA For first response
- Resolution time
- SLA For Resolution
- Close time
- Agent interactions
- Survey results
- Product group
- Support Level
- Country
- Latitude
- Longitude

Sample Data:


,Status,Ticket ID,Priority,Source,Topic,Agent Group,Agent Name,Created time,Expected SLA to resolve,Expected SLA to first response,...,Resolution time,SLA For Resolution,Close time,Agent interactions,Survey results,Product group,Support Level,Country,Latitude,Longitude
0,Closed,TCKT-100000,High,Email,General Inquiry,Security,Khalid Al-Salem,2024-07-04 12:42:00,2024-07-04 14:42:00,2024-07-04 13:12:00,...,2024-07-04 14:30:00,Met,2024-07-04 14:32:00,5,Neutral,Cloud,L3,Oman,25.1856,50.9447
1,Closed,TCKT-100001,High,Chat,Network Issue,Customer Service,Ahmed Al-Sabah,2024-05-23 20:03:00,2024-05-23 22:03:00,2024-05-23 20:33:00,...,2024-05-23 22:00:00,Met,2024-05-23 22:05:00,4,Dissatisfied,Cloud,L2,Qatar,23.2741,55.3867
2,In Progress,TCKT-100002,Low,Phone,General Inquiry,Development,Mohammed Al-Mansoori,2024-04-13 20:51:00,2024-04-14 00:51:00,2024-04-13 21:51:00,...,2024-04-14 00:47:00,Met,2024-04-14 00:51:00,3,Dissatisfied,Software,L1,Bahrain,23.6264,50.1302


# Step 2 — Create Searchable Text + Generate Embeddings + Build FAISS Index

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

In [ ]:
# =========================
# CREATE SEARCHABLE TEXT
# =========================

def create_search_text(row):

    return f"""
    Topic: {row['Topic']}
    Priority: {row['Priority']}
    Status: {row['Status']}
    Country: {row['Country']}
    Product Group: {row['Product group']}
    Support Level: {row['Support Level']}
    Agent Group: {row['Agent Group']}

    Expected Resolve SLA: {row['Expected SLA to resolve']}
    Expected First Response SLA: {row['Expected SLA to first response']}

    Resolution Time: {row['Resolution time']}
    First Response Time: {row['First response time']}

    Survey Results: {row['Survey results']}
    """

df["search_text"] = df.apply(create_search_text, axis=1)

print("Example Search Text:\n")
print(df["search_text"].iloc[0])

Example Search Text:


    Topic: General Inquiry
    Priority: High
    Status: Closed
    Country: Oman
    Product Group: Cloud
    Support Level: L3
    Agent Group: Security

    Expected Resolve SLA: 2024-07-04 14:42:00
    Expected First Response SLA: 2024-07-04 13:12:00

    Resolution Time: 2024-07-04 14:30:00
    First Response Time: 2024-07-04 13:02:00

    Survey Results: Neutral
    


In [ ]:
# =========================
# LOAD EMBEDDING MODEL
# =========================

print("\nLoading embedding model...")

embedding_model = SentenceTransformer(
   "BAAI/bge-base-en-v1.5"
)

print("Embedding model loaded.")


Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded.


In [ ]:
# =========================
# GENERATE EMBEDDINGS
# =========================

texts = df["search_text"].tolist()

embeddings = embedding_model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

embeddings = embeddings.astype("float32")

print("\nEmbedding Shape:", embeddings.shape)


Batches:   0%|          | 0/3125 [00:00<?, ?it/s]


Embedding Shape: (100000, 768)


In [ ]:
# =========================
# BUILD FAISS INDEX
# =========================

dimension = embeddings.shape[1]

faiss_index = faiss.IndexFlatL2(dimension)

faiss_index.add(embeddings)

print("\nFAISS index created.")
print("Total vectors indexed:", faiss_index.ntotal)


FAISS index created.
Total vectors indexed: 100000


# Step 3 — Build and Test Retrieval Function

In [ ]:
# =========================
# RETRIEVAL FUNCTION
# =========================

def retrieve_tickets(query, top_k=10):

    # Encode query
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    # Search FAISS
    distances, indices = faiss_index.search(
        query_embedding,
        top_k
    )

    # Return matching rows
    results = df.iloc[indices[0]].copy()

    results["Similarity_Score"] = distances[0]

    return results

In [ ]:
# =========================
# TEST RETRIEVAL
# =========================

test_query = "VPN not working"

results = retrieve_tickets(test_query)

print(f"\nQuery: {test_query}")
print("\nTop Matches:\n")

display(
    results[
        [
            "Topic",
            "Priority",
            "Status",
            "Agent Group",
            "Country",
            "Similarity_Score"
        ]
    ]
)


Query: VPN not working

Top Matches:



,Topic,Priority,Status,Agent Group,Country,Similarity_Score
51910,Network Issue,Low,Closed,Network Ops,Qatar,0.868803
49136,Network Issue,Low,Closed,Network Ops,Kuwait,0.870026
92376,Network Issue,Low,Closed,IT Support,UAE,0.871042
68604,Network Issue,Low,Closed,Network Ops,UAE,0.872412
73585,Network Issue,Low,Closed,IT Support,UAE,0.872668
80048,Network Issue,Low,Closed,IT Support,Oman,0.872787
38107,Network Issue,Low,Closed,Network Ops,Kuwait,0.873511
23429,Network Issue,Low,Resolved,Security,Kuwait,0.873736
45257,Network Issue,Low,Closed,Network Ops,UAE,0.873782
90312,Network Issue,Low,Closed,Network Ops,Qatar,0.874290


In [ ]:
retrieve_tickets("VPN issue")

,Status,Ticket ID,Priority,Source,Topic,Agent Group,Agent Name,Created time,Expected SLA to resolve,Expected SLA to first response,...,Close time,Agent interactions,Survey results,Product group,Support Level,Country,Latitude,Longitude,search_text,Similarity_Score
63824,Resolved,TCKT-163824,Low,Phone,Network Issue,Security,Mohammed Al-Salem,2024-05-16 17:14:00,2024-05-16 21:14:00,2024-05-16 18:14:00,...,2024-05-16 20:55:00,2,Dissatisfied,Network,L1,UAE,23.137,54.1802,\n Topic: Network Issue\n Priority: Low\...,0.741362
26571,Resolved,TCKT-126571,Medium,Email,Network Issue,Security,Ahmed Al-Khalifa,2024-05-20 21:33:00,2024-05-21 00:33:00,2024-05-20 22:18:00,...,2024-05-21 00:33:00,4,Dissatisfied,Network,L2,Kuwait,22.4118,55.4517,\n Topic: Network Issue\n Priority: Medi...,0.741412
27881,Resolved,TCKT-127881,High,Email,Network Issue,Security,Yousef Al-Hamdan,2024-06-04 15:15:00,2024-06-04 17:15:00,2024-06-04 15:45:00,...,2024-06-04 17:16:00,2,Dissatisfied,Hardware,L2,UAE,22.3318,51.6259,\n Topic: Network Issue\n Priority: High...,0.742587
31815,Resolved,TCKT-131815,Low,Portal,Network Issue,Security,Omar Al-Khalifa,2024-07-22 15:08:00,2024-07-22 19:08:00,2024-07-22 16:08:00,...,2024-07-22 19:09:00,3,Dissatisfied,Network,L1,Kuwait,23.3425,51.1015,\n Topic: Network Issue\n Priority: Low\...,0.743045
19517,Resolved,TCKT-119517,Medium,Portal,Network Issue,Security,Fahad Al-Mansoori,2024-08-03 09:40:00,2024-08-03 12:40:00,2024-08-03 10:25:00,...,2024-08-03 12:32:00,3,Dissatisfied,Hardware,L2,Kuwait,24.0498,53.4587,\n Topic: Network Issue\n Priority: Medi...,0.743065
1892,Resolved,TCKT-101892,Medium,Chat,Network Issue,Security,Omar Al-Hamdan,2024-04-13 02:19:00,2024-04-13 05:19:00,2024-04-13 03:04:00,...,2024-04-13 05:05:00,3,Dissatisfied,Network,L2,Kuwait,22.5939,54.1392,\n Topic: Network Issue\n Priority: Medi...,0.743132
23429,Resolved,TCKT-123429,Low,Chat,Network Issue,Security,Yousef Al-Hamdan,2024-08-06 00:28:00,2024-08-06 04:28:00,2024-08-06 01:28:00,...,2024-08-06 04:21:00,3,Dissatisfied,Network,L3,Kuwait,25.392,50.3936,\n Topic: Network Issue\n Priority: Low\...,0.743185
53241,Resolved,TCKT-153241,Low,Portal,Network Issue,Security,Khalid Al-Nasser,2024-05-29 18:55:00,2024-05-29 22:55:00,2024-05-29 19:55:00,...,2024-05-29 22:53:00,4,Dissatisfied,Hardware,L2,UAE,22.5746,53.8229,\n Topic: Network Issue\n Priority: Low\...,0.743311
45535,Resolved,TCKT-145535,Medium,Portal,Network Issue,Security,Abdullah Al-Farsi,2024-05-10 04:13:00,2024-05-10 07:13:00,2024-05-10 04:58:00,...,2024-05-10 07:09:00,1,Dissatisfied,Hardware,L2,Qatar,22.799,52.9847,\n Topic: Network Issue\n Priority: Medi...,0.743799
15581,Resolved,TCKT-115581,Medium,Email,Network Issue,Security,Khalid Al-Hamdan,2024-05-11 11:53:00,2024-05-11 14:53:00,2024-05-11 12:38:00,...,2024-05-11 14:56:00,3,Dissatisfied,Hardware,L3,Kuwait,22.1948,52.608,\n Topic: Network Issue\n Priority: Medi...,0.743936


In [ ]:
retrieve_tickets("email problem")

,Status,Ticket ID,Priority,Source,Topic,Agent Group,Agent Name,Created time,Expected SLA to resolve,Expected SLA to first response,...,Close time,Agent interactions,Survey results,Product group,Support Level,Country,Latitude,Longitude,search_text,Similarity_Score
89920,Closed,TCKT-189920,Medium,Phone,Network Issue,IT Support,Khalid Al-Farsi,2024-07-26 01:19:00,2024-07-26 04:19:00,2024-07-26 02:04:00,...,2024-07-26 04:03:00,5,Dissatisfied,Software,L1,Saudi Arabia,24.0254,50.1792,\n Topic: Network Issue\n Priority: Medi...,0.707351
17232,Closed,TCKT-117232,Medium,Chat,Network Issue,IT Support,Hassan Al-Mansoori,2024-06-06 05:19:00,2024-06-06 08:19:00,2024-06-06 06:04:00,...,2024-06-06 08:03:00,1,Dissatisfied,Cloud,L2,Saudi Arabia,23.8155,54.2137,\n Topic: Network Issue\n Priority: Medi...,0.709856
28187,Closed,TCKT-128187,Medium,Chat,Network Issue,IT Support,Fahad Al-Khalifa,2024-06-26 07:18:00,2024-06-26 10:18:00,2024-06-26 08:03:00,...,2024-06-26 10:06:00,5,Dissatisfied,Software,L3,Oman,23.1818,53.1299,\n Topic: Network Issue\n Priority: Medi...,0.710185
21190,Closed,TCKT-121190,Medium,Email,Network Issue,IT Support,Ahmed Al-Sabah,2024-04-11 20:39:00,2024-04-11 23:39:00,2024-04-11 21:24:00,...,2024-04-11 23:28:00,1,Dissatisfied,Software,L1,Saudi Arabia,23.3546,54.0084,\n Topic: Network Issue\n Priority: Medi...,0.710677
21233,Closed,TCKT-121233,Medium,Email,Network Issue,Customer Service,Ali Al-Hamdan,2024-04-30 10:46:00,2024-04-30 13:46:00,2024-04-30 11:31:00,...,2024-04-30 13:43:00,5,Dissatisfied,Cloud,L2,Oman,23.6539,50.8955,\n Topic: Network Issue\n Priority: Medi...,0.710832
72619,Closed,TCKT-172619,Medium,Chat,Network Issue,Customer Service,Ahmed Al-Sabah,2024-04-11 17:18:00,2024-04-11 20:18:00,2024-04-11 18:03:00,...,2024-04-11 20:11:00,5,Dissatisfied,Cloud,L3,Saudi Arabia,22.4029,55.4468,\n Topic: Network Issue\n Priority: Medi...,0.711050
16483,Closed,TCKT-116483,Medium,Chat,Network Issue,Customer Service,Hassan Al-Mansoori,2024-07-26 05:18:00,2024-07-26 08:18:00,2024-07-26 06:03:00,...,2024-07-26 08:04:00,3,Dissatisfied,Software,L1,Saudi Arabia,22.5204,50.1213,\n Topic: Network Issue\n Priority: Medi...,0.711067
97799,Closed,TCKT-197799,Medium,Email,Network Issue,Customer Service,Omar Al-Hamdan,2024-04-24 16:19:00,2024-04-24 19:19:00,2024-04-24 17:04:00,...,2024-04-24 19:22:00,2,Dissatisfied,Cloud,L2,Oman,22.7079,53.53,\n Topic: Network Issue\n Priority: Medi...,0.711995
87664,Closed,TCKT-187664,Medium,Chat,Network Issue,IT Support,Ahmed Al-Salem,2024-04-27 02:54:00,2024-04-27 05:54:00,2024-04-27 03:39:00,...,2024-04-27 05:49:00,1,Dissatisfied,Cloud,L2,Oman,23.5559,53.4937,\n Topic: Network Issue\n Priority: Medi...,0.712370
99491,Closed,TCKT-199491,Medium,Portal,Network Issue,IT Support,Khalid Al-Rashid,2024-08-07 16:08:00,2024-08-07 19:08:00,2024-08-07 16:53:00,...,2024-08-07 19:11:00,2,Dissatisfied,Software,L2,Saudi Arabia,24.021,52.6763,\n Topic: Network Issue\n Priority: Medi...,0.712381


In [ ]:
retrieve_tickets("high priority ticket")

,Status,Ticket ID,Priority,Source,Topic,Agent Group,Agent Name,Created time,Expected SLA to resolve,Expected SLA to first response,...,Close time,Agent interactions,Survey results,Product group,Support Level,Country,Latitude,Longitude,search_text,Similarity_Score
45685,Resolved,TCKT-145685,High,Phone,Access Request,Customer Service,Khalid Al-Farsi,2024-08-09 21:12:00,2024-08-09 23:12:00,2024-08-09 21:42:00,...,2024-08-09 23:08:00,1,Neutral,Hardware,L2,Qatar,23.7956,54.6656,\n Topic: Access Request\n Priority: Hig...,0.709689
93569,Resolved,TCKT-193569,High,Email,Access Request,Customer Service,Omar Al-Sabah,2024-08-02 11:43:00,2024-08-02 13:43:00,2024-08-02 12:13:00,...,2024-08-02 13:23:00,4,Neutral,Hardware,L1,Qatar,25.1676,55.0257,\n Topic: Access Request\n Priority: Hig...,0.709806
15566,Resolved,TCKT-115566,High,Phone,Access Request,Customer Service,Khalid Al-Mansoori,2024-04-01 23:57:00,2024-04-02 01:57:00,2024-04-02 00:27:00,...,2024-04-02 01:39:00,2,Neutral,Hardware,L1,Qatar,23.4097,52.1463,\n Topic: Access Request\n Priority: Hig...,0.710724
50459,Resolved,TCKT-150459,High,Chat,Access Request,Customer Service,Omar Al-Rashid,2024-06-24 05:05:00,2024-06-24 07:05:00,2024-06-24 05:35:00,...,2024-06-24 07:06:00,2,Neutral,Hardware,L1,Qatar,23.2747,51.2995,\n Topic: Access Request\n Priority: Hig...,0.712312
47454,Resolved,TCKT-147454,High,Phone,Access Request,Customer Service,Yousef Al-Salem,2024-06-10 06:49:00,2024-06-10 08:49:00,2024-06-10 07:19:00,...,2024-06-10 08:39:00,5,Neutral,Hardware,L1,UAE,24.7045,51.3841,\n Topic: Access Request\n Priority: Hig...,0.713916
76131,Resolved,TCKT-176131,High,Email,Access Request,Customer Service,Hassan Al-Mansoori,2024-04-04 08:54:00,2024-04-04 10:54:00,2024-04-04 09:24:00,...,2024-04-04 10:45:00,2,Neutral,Hardware,L2,Saudi Arabia,24.5023,55.4688,\n Topic: Access Request\n Priority: Hig...,0.714758
27701,Resolved,TCKT-127701,High,Portal,Access Request,Customer Service,Khalid Al-Mansoori,2024-04-10 03:10:00,2024-04-10 05:10:00,2024-04-10 03:40:00,...,2024-04-10 04:57:00,2,Neutral,Network,L2,Saudi Arabia,23.9139,55.9467,\n Topic: Access Request\n Priority: Hig...,0.714832
87945,Resolved,TCKT-187945,Critical,Chat,Access Request,IT Support,Hassan Al-Salem,2024-08-03 09:09:00,2024-08-03 10:09:00,2024-08-03 09:19:00,...,2024-08-03 10:05:00,1,Neutral,Hardware,L2,Qatar,24.2852,55.4829,\n Topic: Access Request\n Priority: Cri...,0.715107
33489,Resolved,TCKT-133489,High,Email,Access Request,Customer Service,Omar Al-Nasser,2024-07-07 03:31:00,2024-07-07 05:31:00,2024-07-07 04:01:00,...,2024-07-07 05:27:00,3,Neutral,Software,L1,UAE,23.1924,51.8567,\n Topic: Access Request\n Priority: Hig...,0.716213
95249,Resolved,TCKT-195249,High,Email,Access Request,Customer Service,Hassan Al-Rashid,2024-06-08 00:58:00,2024-06-08 02:58:00,2024-06-08 01:28:00,...,2024-06-08 02:56:00,5,Neutral,Network,L1,UAE,25.1958,53.0212,\n Topic: Access Request\n Priority: Hig...,0.716377


In [ ]:
retrieve_tickets("network outage")

,Status,Ticket ID,Priority,Source,Topic,Agent Group,Agent Name,Created time,Expected SLA to resolve,Expected SLA to first response,...,Close time,Agent interactions,Survey results,Product group,Support Level,Country,Latitude,Longitude,search_text,Similarity_Score
40229,Closed,TCKT-140229,Critical,Portal,Hardware Failure,Network Ops,Ali Al-Khalifa,2024-08-04 20:12:00,2024-08-04 21:12:00,2024-08-04 20:22:00,...,2024-08-04 21:13:00,1,Neutral,Network,L3,Kuwait,23.9625,50.4969,\n Topic: Hardware Failure\n Priority: C...,0.632215
33176,Closed,TCKT-133176,Medium,Email,Hardware Failure,Network Ops,Hassan Al-Sabah,2024-04-13 15:59:00,2024-04-13 18:59:00,2024-04-13 16:44:00,...,2024-04-13 19:00:00,5,Neutral,Network,L1,Kuwait,25.8387,55.978,\n Topic: Hardware Failure\n Priority: M...,0.632318
65611,Closed,TCKT-165611,Low,Phone,Hardware Failure,Network Ops,Fahad Al-Sabah,2024-05-23 05:20:00,2024-05-23 09:20:00,2024-05-23 06:20:00,...,2024-05-23 09:14:00,2,Dissatisfied,Network,L1,Kuwait,22.7339,53.8663,\n Topic: Hardware Failure\n Priority: L...,0.633524
47866,Closed,TCKT-147866,Medium,Phone,Hardware Failure,Network Ops,Hassan Al-Nasser,2024-05-03 17:17:00,2024-05-03 20:17:00,2024-05-03 18:02:00,...,2024-05-03 20:00:00,4,Neutral,Network,L3,Kuwait,22.6906,51.9097,\n Topic: Hardware Failure\n Priority: M...,0.633635
32650,Closed,TCKT-132650,Low,Phone,Hardware Failure,Network Ops,Saif Al-Farsi,2024-07-11 03:25:00,2024-07-11 07:25:00,2024-07-11 04:25:00,...,2024-07-11 07:23:00,1,Dissatisfied,Network,L3,Kuwait,22.0214,50.6486,\n Topic: Hardware Failure\n Priority: L...,0.633726
8697,Closed,TCKT-108697,Medium,Portal,Hardware Failure,Network Ops,Hassan Al-Rashid,2024-06-25 08:46:00,2024-06-25 11:46:00,2024-06-25 09:31:00,...,2024-06-25 11:28:00,1,Dissatisfied,Network,L3,Kuwait,25.183,52.893,\n Topic: Hardware Failure\n Priority: M...,0.633775
31797,Closed,TCKT-131797,Medium,Phone,Hardware Failure,Network Ops,Yousef Al-Nasser,2024-06-05 22:54:00,2024-06-06 01:54:00,2024-06-05 23:39:00,...,2024-06-06 01:52:00,2,Neutral,Network,L3,Kuwait,22.384,55.1003,\n Topic: Hardware Failure\n Priority: M...,0.634074
97726,Closed,TCKT-197726,Medium,Email,Hardware Failure,Network Ops,Yousef Al-Salem,2024-06-20 03:39:00,2024-06-20 06:39:00,2024-06-20 04:24:00,...,2024-06-20 06:32:00,1,Dissatisfied,Network,L1,Kuwait,24.369,52.9721,\n Topic: Hardware Failure\n Priority: M...,0.634360
83855,Closed,TCKT-183855,Medium,Portal,Hardware Failure,Network Ops,Ali Al-Salem,2024-08-06 16:49:00,2024-08-06 19:49:00,2024-08-06 17:34:00,...,2024-08-06 19:39:00,2,Dissatisfied,Network,L1,Kuwait,25.4476,53.8372,\n Topic: Hardware Failure\n Priority: M...,0.635134
32179,Closed,TCKT-132179,Medium,Chat,Hardware Failure,Network Ops,Omar Al-Mansoori,2024-04-18 05:18:00,2024-04-18 08:18:00,2024-04-18 06:03:00,...,2024-04-18 08:07:00,5,Dissatisfied,Network,L2,Kuwait,23.4034,53.3537,\n Topic: Hardware Failure\n Priority: M...,0.635263


# Step 4 — Analytics Engine

In [ ]:
# =========================
# ANALYTICS FUNCTIONS
# =========================

def dataset_summary():
  return f"""
Total Tickets: {len(df)}

Unique Topics: {df['Topic'].nunique()}

Countries Covered: {df['Country'].nunique()}

Agent Groups: {df['Agent Group'].nunique()}
"""


def status_counts():

    return df["Status"].value_counts()


def priority_counts():

    return df["Priority"].value_counts()


def country_counts(top_n=10):

    return df["Country"].value_counts().head(top_n)


def tickets_by_country(country):

    result = df[
        df["Country"].str.lower() == country.lower()
    ]

    return result[
        [
            "Ticket ID",
            "Topic",
            "Priority",
            "Status"
        ]
    ]


def tickets_by_priority(priority):

    result = df[
        df["Priority"].str.lower() == priority.lower()
    ]

    return result[
        [
            "Ticket ID",
            "Topic",
            "Status",
            "Country"
        ]
    ]


def open_tickets():

    result = df[
        df["Status"].str.lower().str.contains("open")
    ]

    return result[
        [
            "Ticket ID",
            "Topic",
            "Priority",
            "Country"
        ]
    ]

def open_ticket_count():

    return len(
        df[df["Status"].str.lower()=="open"]
    )


def closed_ticket_count():

    return len(
        df[df["Status"].str.lower()=="closed"]
    )


def top_countries():

    return (
        df["Country"]
        .value_counts()
        .head(10)
    )


def top_topics():

    return (
        df["Topic"]
        .value_counts()
        .head(10)
    )


def top_agent_groups():

    return (
        df["Agent Group"]
        .value_counts()
        .head(10)
    )

def sla_summary():

    return f"""
Total Tickets: {len(df)}

Average Resolution Time:
{df['Resolution time'].head(1000).astype(str).mode()[0]}

Average First Response Time:
{df['First response time'].head(1000).astype(str).mode()[0]}
"""

In [ ]:
HELP_MESSAGE = """
Hello! I am your ITSM Assistant.

I can analyze our ticket data for quick stats, or search historical logs to help diagnose technical issues.

📊 Ask for Statistics & Reports:
• "How many open tickets?" or "How many closed tickets?"
• "What are the top countries?"
• "Show me top topics"
• "Show me agent statistics"
• "What are the top priorities?"
• "Give me an SLA summary"
• "Show me a dataset summary"

🔍 Search & Troubleshoot IT Issues:
• "VPN is not working"
• "Network outage"
• "Email login problem"

Just type your question below and I'll get to work!
"""

In [ ]:
def is_safe(query):

    blocked = [
        "hack",
        "malware",
        "virus",
        "sql injection",
        "exploit"
    ]

    query = query.lower()

    return not any(
        word in query
        for word in blocked
    )

In [ ]:
def out_of_scope(query):
    # Expanded list to catch variations and analytics terms
    allowed_keywords = [
        "ticket", "tickets", "incident", "vpn", "network", "email", "sla",
        "country", "countries", "priority", "priorities", "support",
        "agent", "agents", "topic", "topics", "status", "open", "closed",
        "how many", "top", "summary"
    ]

    query = query.lower()

    return not any(
        word in query
        for word in allowed_keywords
    )

In [ ]:
print(dataset_summary())


Total Tickets: 100000

Unique Topics: 5

Countries Covered: 6

Agent Groups: 5



In [ ]:
status_counts()

,count
Status,
Resolved,20134
In Progress,20123
Closed,20015
New,20014
Open,19714


In [ ]:
priority_counts()

,count
Priority,
Medium,25117
Critical,25045
Low,25014
High,24824


In [ ]:
country_counts()

,count
Country,
Qatar,17005
Kuwait,16640
Saudi Arabia,16635
UAE,16600
Bahrain,16580
Oman,16540


In [ ]:
tickets_by_country("Qatar")

,Ticket ID,Topic,Priority,Status
1,TCKT-100001,Network Issue,High,Closed
4,TCKT-100004,Hardware Failure,Critical,Closed
5,TCKT-100005,Network Issue,Medium,In Progress
10,TCKT-100010,General Inquiry,Critical,Closed
13,TCKT-100013,Network Issue,High,New
...,...,...,...,...
99981,TCKT-199981,Network Issue,Low,New
99986,TCKT-199986,Access Request,Low,In Progress
99990,TCKT-199990,Hardware Failure,High,New
99995,TCKT-199995,Network Issue,Low,Closed


In [ ]:
tickets_by_priority("High")

,Ticket ID,Topic,Status,Country
0,TCKT-100000,General Inquiry,Closed,Oman
1,TCKT-100001,Network Issue,Closed,Qatar
7,TCKT-100007,General Inquiry,In Progress,UAE
13,TCKT-100013,Network Issue,New,Qatar
15,TCKT-100015,Software Bug,New,Saudi Arabia
...,...,...,...,...
99988,TCKT-199988,Access Request,In Progress,Bahrain
99989,TCKT-199989,General Inquiry,In Progress,Bahrain
99990,TCKT-199990,Hardware Failure,New,Qatar
99998,TCKT-199998,Hardware Failure,In Progress,Qatar


In [ ]:
open_tickets()

,Ticket ID,Topic,Priority,Country
8,TCKT-100008,Access Request,Medium,Saudi Arabia
9,TCKT-100009,Network Issue,Critical,Kuwait
12,TCKT-100012,Hardware Failure,Critical,Kuwait
23,TCKT-100023,Hardware Failure,Critical,Oman
24,TCKT-100024,Hardware Failure,High,Kuwait
...,...,...,...,...
99978,TCKT-199978,Network Issue,Medium,Kuwait
99979,TCKT-199979,Software Bug,Critical,Kuwait
99980,TCKT-199980,Access Request,Low,Saudi Arabia
99984,TCKT-199984,Hardware Failure,High,Kuwait


# Step 5 — Query Router

In [ ]:
# =========================
# QUERY ROUTER
# =========================

analytics_keywords = [
    "how many", "total", "count", "summary", "statistics", "distribution", "breakdown",
    "status", "open", "closed", "progress", "resolved", "new",
    "priority", "priorities", "source", "channel", "medium",
    "topic", "topics", "agent group", "agent groups", "agent name", "top agent", "agents",
    "sla", "breach", "met", "interaction", "interactions", "survey", "satisfaction",
    "csat", "product", "products", "level", "tier", "l1", "l2", "l3",
    "country", "countries", "location"
]

def analytics_response(query):
    q = query.lower()

    # 1. TOTAL TICKET VOLUME
    if "how many tickets" in q or "total tickets" in q or "ticket count" in q:
        return f"Total Tickets in Dataset: {len(df)}"

    # 2. STATUS BREAKDOWNS
    if "open" in q and "count" in q:
        count = len(df[df["Status"].str.lower() == "open"])
        return f"Total Open Tickets: {count}"
    if "closed" in q and "count" in q:
        count = len(df[df["Status"].str.lower() == "closed"])
        return f"Total Closed Tickets: {count}"
    if "progress" in q:
        count = len(df[df["Status"].str.lower() == "in progress"])
        return f"Tickets currently In Progress: {count}"
    if "resolved" in q:
        count = len(df[df["Status"].str.lower() == "resolved"])
        return f"Total Resolved Tickets: {count}"
    if "status distribution" in q or "status breakdown" in q or "all status" in q:
        return f"Status Breakdown:\n{df['Status'].value_counts().to_string()}"

    # 3. PRIORITY BREAKDOWN
    if "priority" in q or "priorities" in q:
        return f"Priority Distribution:\n{df['Priority'].value_counts().to_string()}"

    # 4. SOURCE / CHANNEL ANALYSIS (Email, Chat, Phone)
    if "source" in q or "channel" in q or "medium" in q:
        return f"Ticket Source Distribution:\n{df['Source'].value_counts().to_string()}"

    # 5. TOPIC BREAKDOWN
    if "topic" in q or "topics" in q:
        return f"Top Ticket Topics:\n{df['Topic'].value_counts().head(10).to_string()}"

    # 6. AGENT GROUPS
    if "agent group" in q or "agent groups" in q:
        return f"Agent Group Workload:\n{df['Agent Group'].value_counts().to_string()}"

    # 7. AGENT PERFORMANCE / NAMES
    if "agent name" in q or "top agent" in q or "active agent" in q:
        return f"Most Active Agents (Top 10):\n{df['Agent Name'].value_counts().head(10).to_string()}"

    # 8. SLA PERFORMANCE (Met vs Breached)
    if "sla for resolution" in q or "sla performance" in q or "sla breakdown" in q:
        return f"SLA Resolution Performance:\n{df['SLA For Resolution'].value_counts().to_string()}"
    if "sla" in q:  # Catch-all for basic SLA metrics
        return sla_summary()

    # 9. AGENT INTERACTIONS (Average touches per ticket)
    if "interaction" in q or "touchpoint" in q:
        try:
            avg_interactions = pd.to_numeric(df["Agent interactions"], errors='coerce').mean()
            return f"Average Agent Interactions per Ticket: {avg_interactions:.2f}"
        except:
            return f"Agent Interactions Breakdown:\n{df['Agent interactions'].value_counts().head(5).to_string()}"

    # 10. SURVEY RESULTS / CUSTOMER SATISFACTION (CSAT)
    if "survey" in q or "satisfaction" in q or "csat" in q:
        return f"Customer Survey Results Breakdown:\n{df['Survey results'].value_counts().to_string()}"

    # 11. PRODUCT GROUPS
    if "product" in q or "products" in q:
        return f"Product Group Distribution:\n{df['Product group'].value_counts().to_string()}"

    # 12. SUPPORT LEVELS (L1, L2, L3)
    if "support level" in q or "level" in q or "tier" in q or "l1" in q or "l2" in q or "l3" in q:
        return f"Support Level Tier Breakdown:\n{df['Support Level'].value_counts().to_string()}"

    # 13. COUNTRIES / GEOGRAPHY
    if "country" in q or "countries" in q or "location" in q:
        return f"Top 10 Countries by Ticket Volume:\n{df['Country'].value_counts().head(10).to_string()}"

    # 14. DATASET SUMMARY (Catch-all for general statistics)
    if "summary" in q or "statistics" in q or "stats" in q:
        return dataset_summary()

    # Insert this at the very bottom of analytics_response() right before the final return:
    for status in df['Status'].unique():
      if status.lower() in q:
        count = len(df[df["Status"].str.lower() == status.lower()])
        return f"Total {status} Tickets: {count}"

    return "Analytics query recognized, but I couldn't isolate the exact column. Try using specific terms like 'status distribution', 'agent groups', or 'survey results'."

# =========================
# RETRIEVAL RESPONSE
# =========================

def retrieval_response(query):

    results = retrieve_tickets(query, top_k=10)

    answer = "Top Similar Tickets:\n\n"

    for _, row in results.iterrows():

        answer += (
            f"Topic: {row['Topic']}\n"
            f"Priority: {row['Priority']}\n"
            f"Status: {row['Status']}\n"
            f"Country: {row['Country']}\n"
            f"----------------------\n"
        )

    return answer


# =========================
# MAIN CHATBOT
# =========================

def chatbot(query):
   if not is_safe(query):
       return "I can only assist with ITSM ticket analysis."

   if out_of_scope(query):
       return HELP_MESSAGE

   q = query.lower().strip()

   greetings = ["hi", "hello", "hey", "help"]
   if any(g == q for g in greetings):
       return HELP_MESSAGE

   # Improved routing directly inside chatbot for common analytics misses
   if "how many open" in q or ("open" in q and "count" in q):
       return f"Total Open Tickets: {open_ticket_count()}"

   if "how many closed" in q or ("closed" in q and "count" in q):
       return f"Total Closed Tickets: {closed_ticket_count()}"

   if any(keyword in q for keyword in analytics_keywords):
       return analytics_response(query)

   return rag_response(query)

In [ ]:
print(chatbot("survey"))


Hello! I am your ITSM Assistant.

I can analyze our ticket data for quick stats, or search historical logs to help diagnose technical issues.

📊 Ask for Statistics & Reports:
• "How many open tickets?" or "How many closed tickets?"
• "What are the top countries?"
• "Show me top topics"
• "Show me agent statistics"
• "What are the top priorities?"
• "Give me an SLA summary"
• "Show me a dataset summary"

🔍 Search & Troubleshoot IT Issues:
• "VPN is not working"
• "Network outage"
• "Email login problem"

Just type your question below and I'll get to work!



In [ ]:
print(chatbot("Show status distribution"))

Status Breakdown:
Status
Resolved       20134
In Progress    20123
Closed         20015
New            20014
Open           19714


In [ ]:
print(chatbot("Show high priority tickets"))

Priority Distribution:
Priority
Medium      25117
Critical    25045
Low         25014
High        24824


# Step 6 — Add LLM Summarization

In [ ]:
# =========================
# LOAD LLM
# =========================

from transformers import pipeline
import torch

print("Loading LLM...")

generator = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    device_map="auto",
    torch_dtype=torch.float16
)

print("LLM Loaded.")

Loading LLM...


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

LLM Loaded.


In [ ]:
def rag_response(query):
    # Retrieve top 5 matching historical records
    results = retrieve_tickets(query, top_k=10)

    # SMART GUARDRAIL: Check the best semantic distance match
    # Completely unrelated queries will have a massive distance score (usually > 1.4)
    best_match_score = results["Similarity_Score"].iloc[0]
    if best_match_score > 1.4:
        return HELP_MESSAGE

    context = ""
    for _, row in results.iterrows():
        context += f"- Topic: {row['Topic']}, Priority: {row['Priority']}, Status: {row['Status']}, Group: {row['Agent Group']}\n"

    # TinyLlama friendly system format
    prompt = f"""<|system|>
You are a helpful IT Service Management assistant. Look at the historical tickets and provide a short, 3-sentence summary of how similar issues are usually classified and handled. Do not repeat the prompt.
</s>
<|user|>
User Issue: "{query}"

Historical Tickets:
{context}
</s>
<|assistant|>
"""

    output = generator(
        prompt,
        max_new_tokens=300,
        do_sample=True,
        temperature=0.3,
        repetition_penalty=1.2
    )

    raw_response = output[0]["generated_text"]

    if "<|assistant|>" in raw_response:
        cleaned_response = raw_response.split("<|assistant|>")[-1].strip()
    else:
        cleaned_response = DEFAULT_RAG_RESPONSE + context.strip()

    if not cleaned_response or len(cleaned_response) < 10:
        return DEFAULT_RAG_RESPONSE + context.strip()

    return cleaned_response

In [ ]:
def chatbot(query):

    if not is_safe(query):

        return (
            "I can only assist with "
            "ITSM ticket analysis."
        )

    if out_of_scope(query):

        return HELP_MESSAGE

    q = query.lower()

    greetings = [
        "hi",
        "hello",
        "hey"
    ]

    if any(g in q for g in greetings):

        return HELP_MESSAGE

    if any(
        keyword in q
        for keyword in analytics_keywords
    ):

        return analytics_response(query)

    return rag_response(query)

In [ ]:
print(chatbot("VPN not connecting"))

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'repetition_penalty', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


To help resolve this issue, you can start by checking if there is any network connectivity issue between your device and the VPN server. You may need to check for firewall settings or update software on both devices to ensure that they are working correctly. If none of these steps work, it could be an issue with the VPN provider's servers or connection parameters. In such cases, you will need to contact their support team directly to troubleshoot the issue further. Additionally, consider using a different VPN service in case one of the above solutions does not solve the problem.


In [ ]:
# print(chatbot("Outlook email issue"))

[transformers] Both `max_new_tokens` (=300) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


To summarize the historical tickets related to Outlook email issues, we can categorize them as follows:

1. Topic: Network Issue, Priority: Medium, Status: Closed, Group: IT Support - These tickets were closed due to the resolution of an issue with network connectivity or performance. The group assigned these tickets was IT Support.
2. Topic: Network Issue, Priority: Critical, Status: Resolved, Group: IT Support - These tickets were resolved by addressing the root cause of the problem. The group assigned these tickets was IT Support.
3. Topic: Network Issue, Priority: Medium, Status: Closed, Group: IT Support - These tickets were closed because they did not require any action from IT Support. However, if there is ongoing maintenance required for the affected system, it would be addressed in future updates. The group assigned these tickets was IT Support.
4. Topic: Network Issue, Priority: Low, Status: In Progress, Group: IT Support - These tickets are still being worked on by the IT te

In [ ]:
# print(chatbot("Network outage"))

[transformers] Both `max_new_tokens` (=300) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


1. Similar issues with hardware failures tend to be categorized under the priority level of medium or low depending on their severity. For instance, if there is a failure in one server that affects all servers in the network, it would fall into the medium category since it has a lower impact than other instances where multiple servers were affected by the same issue.
2. The group responsible for managing these types of incidents typically includes engineers from different departments such as network operations, infrastructure management, and system administration. They work together to identify root causes and fix them before they escalate beyond the critical status.
3. In some cases, the incident may require an urgent response due to its severe nature. In those situations, the team will prioritize the issue based on its potential impact on the business and take immediate action to mitigate any damage caused. This could involve implementing measures like rolling back data or restoring 

In [ ]:
import gradio as gr
import matplotlib.pyplot as plt
import io
from PIL import Image

# ==========================================
# PLOTTING FUNCTIONS FOR THE DASHBOARD
# ==========================================

def plot_status_distribution():
    plt.figure(figsize=(6, 4))
    counts = df["Status"].value_counts()
    counts.plot(kind="bar", color=["#4F46E5", "#10B981", "#F59E0B", "#EF4444"][:len(counts)])
    plt.title("Ticket Distribution by Status")
    plt.ylabel("Number of Tickets")
    plt.xticks(rotation=45)
    plt.tight_layout()

    # Save plot to an image buffer
    buf = io.BytesIO()
    plt.savefig(buf, format="png", dpi=150)
    buf.seek(0)
    plt.close()
    return Image.open(buf)

def plot_priority_distribution():
    plt.figure(figsize=(6, 4))
    counts = df["Priority"].value_counts()
    counts.plot(kind="pie", autopct="%1.1f%%", colors=["#EF4444", "#F59E0B", "#3B82F6", "#10B981"][:len(counts)])
    plt.title("Tickets by Priority")
    plt.ylabel("")
    plt.tight_layout()

    buf = io.BytesIO()
    plt.savefig(buf, format="png", dpi=150)
    buf.seek(0)
    plt.close()
    return Image.open(buf)

def plot_top_countries():
    plt.figure(figsize=(8, 4))
    counts = df["Country"].value_counts().head(10)
    counts.plot(kind="barh", color="#2563EB").invert_yaxis()
    plt.title("Top 10 Countries by Ticket Volume")
    plt.xlabel("Number of Tickets")
    plt.tight_layout()

    buf = io.BytesIO()
    plt.savefig(buf, format="png", dpi=150)
    buf.seek(0)
    plt.close()
    return Image.open(buf)

# Refresh dashboard stats dynamically
def load_dashboard_stats():
    summary_text = dataset_summary()
    status_img = plot_status_distribution()
    priority_img = plot_priority_distribution()
    country_img = plot_top_countries()
    return summary_text, status_img, priority_img, country_img


# ==========================================
# GRADIO INTERFACE EXECUTION
# ==========================================

print("Launching UI with Analytics Dashboard...")

# FIXED: 'title' belongs here to define the web page tab name
with gr.Blocks(title="ITSM InsightBot") as demo:
    gr.Markdown("# 📊 ITSM InsightBot")
    gr.Markdown("Switch between the AI Copilot and the Live Metrics Dashboard using the tabs below.")

    with gr.Tabs():

        # TAB 1: Native ChatInterface
        with gr.TabItem("AI Ticket Assistant"):

            def chat_wrapper(message, history):
                return chatbot(message)

            gr.ChatInterface(
                fn=chat_wrapper,
                examples=[
                    "How many open tickets?",
                    "What are the top countries?",
                    "Show me agent statistics",
                    "VPN is not working"
                ]
            )

        # TAB 2: The Dashboard
        with gr.TabItem("📈 Analytics Dashboard"):
            with gr.Row():
                refresh_btn = gr.Button("Refresh Dashboard Data", variant="primary")

            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown("### 📋 High-Level Overview")
                    summary_output = gr.Textbox(
                        value=dataset_summary(),
                        label="Dataset Metrics",
                        lines=8,
                        interactive=False
                    )

                with gr.Column(scale=2):
                    gr.Markdown("### 📊 Status & Priority Breakdown")
                    with gr.Row():
                        status_plot = gr.Image(value=plot_status_distribution(), label="Status Counts", interactive=False)
                        priority_plot = gr.Image(value=plot_priority_distribution(), label="Priority Breakdown", interactive=False)

            with gr.Row():
                with gr.Column():
                    gr.Markdown("### 🌍 Geographic Ticket Volume")
                    country_plot = gr.Image(value=plot_top_countries(), label="Top Countries", interactive=False)

            refresh_btn.click(
                fn=load_dashboard_stats,
                inputs=[],
                outputs=[summary_output, status_plot, priority_plot, country_plot]
            )

# FIXED: Passed 'theme' here to satisfy modern Gradio version compliance requirements
demo.launch(share=True, debug=True, theme="soft")

Launching UI with Analytics Dashboard...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://9928c7c0db316e4743.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
[transformers] Both `max_new_tokens` (=300) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/